In [11]:
"""
Descarga automática de establecimientos MINEDUC + Análisis de calidad de datos.
- Verifica si ya existe el CSV unificado y pregunta si usarlo.
- Si no existe o el usuario decide no usarlo, descarga todo (headless).
- Realiza un diagnóstico completo de los datos.
"""

import os
import time
import glob
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException

# ============================================================================
# CONFIGURACIÓN
# ============================================================================
BASE_URL = "https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/"

DEPARTAMENTOS = [
    "ALTA VERAPAZ", "BAJA VERAPAZ", "CHIMALTENANGO", "CHIQUIMULA",
    "CIUDAD CAPITAL", "EL PROGRESO", "ESCUINTLA", "GUATEMALA",
    "HUEHUETENANGO", "IZABAL", "JALAPA", "JUTIAPA", "PETEN",
    "QUETZALTENANGO", "QUICHE", "RETALHULEU", "SACATEPEQUEZ",
    "SAN MARCOS", "SANTA ROSA", "SOLOLA", "SUCHITEPEQUEZ",
    "TOTONICAPAN", "ZACAPA",
]

NIVELES = ["BASICO", "DIVERSIFICADO"]

CARPETA_DESCARGAS = os.path.join(os.getcwd(), "descargas")
CSV_SALIDA = "mineduc_establecimientos_unificado.csv"

# ============================================================================
# FUNCIONES AUXILIARES (descarga)
# ============================================================================

def encontrar_select_por_nombre(driver, nombre_parcial):
    selects = driver.find_elements(By.TAG_NAME, "select")
    for sel in selects:
        name = sel.get_attribute("name")
        if name and nombre_parcial in name:
            return sel
    return None

def encontrar_boton_por_src(driver, src_parcial):
    inputs = driver.find_elements(By.XPATH, "//input[@type='image']")
    for inp in inputs:
        src = inp.get_attribute("src")
        if src and src_parcial in src:
            return inp
    return None

def hacer_scroll_hasta_elemento(driver, elemento):
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", elemento)
    time.sleep(0.5)

def clic_seguro(driver, elemento):
    try:
        elemento.click()
    except:
        from selenium.webdriver.common.action_chains import ActionChains
        ActionChains(driver).move_to_element(elemento).click().perform()

def esperar_descarga(carpeta, timeout=30):
    archivos_iniciales = set(glob.glob(os.path.join(carpeta, "*.xls")))
    start = time.time()
    while time.time() - start < timeout:
        archivos_actuales = set(glob.glob(os.path.join(carpeta, "*.xls")))
        nuevos = archivos_actuales - archivos_iniciales
        if nuevos:
            return max(nuevos, key=os.path.getctime)
        time.sleep(1)
    raise TimeoutError("No se detectó la descarga del archivo .xls")

def leer_tabla_desde_archivo_xls(ruta_archivo):
    with open(ruta_archivo, 'r', encoding='latin1') as f:
        contenido = f.read()
    tablas = pd.read_html(contenido)
    if not tablas:
        raise ValueError("No se encontraron tablas en el archivo")
    df = max(tablas, key=lambda t: t.shape[0])
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)
    df = df.dropna(how='all')
    return df

def descargar_departamento_nivel(driver, departamento, nivel):
    driver.get(BASE_URL)
    wait = WebDriverWait(driver, 15)

    select_depto = encontrar_select_por_nombre(driver, "cmbDepartamento")
    if not select_depto:
        raise Exception("No se encontró el select de Departamento")
    Select(select_depto).select_by_visible_text(departamento)
    time.sleep(1.5)

    select_nivel = encontrar_select_por_nombre(driver, "cmbNivel")
    if not select_nivel:
        raise Exception("No se encontró el select de Nivel")
    Select(select_nivel).select_by_visible_text(nivel)
    time.sleep(0.5)

    boton_buscar = driver.find_element(By.XPATH, "//input[@type='image' and contains(@src, 'seleccionar_estab.gif')]")
    hacer_scroll_hasta_elemento(driver, boton_buscar)
    clic_seguro(driver, boton_buscar)
    time.sleep(4)

    try:
        wait.until(EC.presence_of_element_located((By.XPATH, "//table")))
    except TimeoutException:
        print(f"  Sin resultados para {departamento} - {nivel}")
        return None

    boton_excel = encontrar_boton_por_src(driver, "export_excel.gif")
    if not boton_excel:
        try:
            boton_excel = driver.find_element(By.XPATH, "//*[contains(text(),'Generar Archivo de Excel')]")
        except:
            pass
    if not boton_excel:
        raise Exception("No se encontró el botón de Excel")

    hacer_scroll_hasta_elemento(driver, boton_excel)
    wait.until(EC.element_to_be_clickable(boton_excel))
    boton_excel.click()

    ruta_archivo = esperar_descarga(CARPETA_DESCARGAS, timeout=30)
    nuevo_nombre = f"{departamento}_{nivel}.xls"
    nueva_ruta = os.path.join(CARPETA_DESCARGAS, nuevo_nombre)
    if os.path.exists(nueva_ruta):
        base, ext = os.path.splitext(nuevo_nombre)
        nueva_ruta = os.path.join(CARPETA_DESCARGAS, f"{base}_{int(time.time())}{ext}")
    os.rename(ruta_archivo, nueva_ruta)
    print(f"  Descargado: {os.path.basename(nueva_ruta)}")
    return nueva_ruta

# ============================================================================
# FUNCIÓN PARA DESCARGAR Y UNIFICAR (si no se usa CSV existente)
# ============================================================================

def descargar_y_unificar():
    os.makedirs(CARPETA_DESCARGAS, exist_ok=True)

    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    prefs = {
        "download.default_directory": CARPETA_DESCARGAS,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    }
    options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(options=options)
    archivos_descargados = []

    try:
        print("Iniciando descarga de todos los departamentos (Básico y Diversificado)...")
        for depto in DEPARTAMENTOS:
            for nivel in NIVELES:
                try:
                    print(f"Procesando {depto} - {nivel}...")
                    ruta = descargar_departamento_nivel(driver, depto, nivel)
                    if ruta:
                        archivos_descargados.append(ruta)
                except Exception as e:
                    print(f"  ERROR: {e}")
                time.sleep(2)

        print(f"\nArchivos descargados: {len(archivos_descargados)}")
        if not archivos_descargados:
            raise SystemExit("No se descargó ningún archivo.")

        dataframes = []
        for archivo in archivos_descargados:
            try:
                df = leer_tabla_desde_archivo_xls(archivo)
                nombre_base = os.path.basename(archivo).replace(".xls", "")
                partes = nombre_base.split("_")
                if len(partes) >= 2:
                    df["DEPARTAMENTO_CONSULTADO"] = partes[0]
                    df["NIVEL_CONSULTADO"] = partes[1]
                dataframes.append(df)
                print(f"Leído: {os.path.basename(archivo)} -> {len(df)} filas")
            except Exception as e:
                print(f"Error al leer {archivo}: {e}")

        if not dataframes:
            raise SystemExit("No se pudieron leer los archivos.")

        df_unificado = pd.concat(dataframes, ignore_index=True)
        print(f"\nTotal filas (antes de deduplicar): {len(df_unificado)}")

        # Limpieza
        df_unificado.columns = [str(c).strip().upper() for c in df_unificado.columns]

        if "CODIGO" in df_unificado.columns:
            df_unificado = df_unificado[df_unificado["CODIGO"].notna()]
            df_unificado = df_unificado[df_unificado["CODIGO"].astype(str).str.strip() != ""]

        categoricas = ["DEPARTAMENTO", "MUNICIPIO", "NIVEL", "SECTOR", "AREA",
                       "STATUS", "MODALIDAD", "JORNADA", "PLAN", "DEPARTAMENTAL"]
        for col in categoricas:
            if col in df_unificado.columns:
                df_unificado[col] = df_unificado[col].astype(str).str.upper().str.strip()
                df_unificado[col] = df_unificado[col].replace({"NAN": pd.NA, "NONE": pd.NA, "---": pd.NA})

        if "TELEFONO" in df_unificado.columns:
            df_unificado["TELEFONO"] = df_unificado["TELEFONO"].astype(str).str.replace(r"\.0$", "", regex=True)
            df_unificado["TELEFONO"] = df_unificado["TELEFONO"].replace({"nan": pd.NA, "<NA>": pd.NA})

        if "CODIGO" in df_unificado.columns:
            antes = len(df_unificado)
            df_unificado["_no_nulos"] = df_unificado.notna().sum(axis=1)
            df_unificado = df_unificado.sort_values("_no_nulos", ascending=False).drop_duplicates(subset=["CODIGO"]).drop(columns="_no_nulos")
            print(f"Duplicados eliminados por CODIGO: {antes - len(df_unificado)}")

        print(f"Total filas final: {len(df_unificado)}")

        # Guardar CSV
        df_unificado.to_csv(CSV_SALIDA, index=False, encoding="utf-8-sig")
        print(f"\nCSV guardado: {CSV_SALIDA} ({len(df_unificado)} filas, {len(df_unificado.columns)} columnas)")

        # Eliminar archivos .xls temporales
        print("\nEliminando archivos .xls descargados para ahorrar espacio...")
        for archivo in archivos_descargados:
            try:
                os.remove(archivo)
            except Exception as e:
                print(f"  No se pudo eliminar {archivo}: {e}")
        try:
            os.rmdir(CARPETA_DESCARGAS)
        except OSError:
            pass

        return df_unificado

    finally:
        driver.quit()

# ============================================================================
# ANÁLISIS DE DIAGNÓSTICO DE DATOS
# ============================================================================

def diagnosticar_datos(df):
    print("\n" + "="*80)
    print("DIAGNÓSTICO DE CALIDAD DE DATOS")
    print("="*80 + "\n")

    # a. Número de registros y variables
    print("a. Número de registros y variables:")
    print(f"   - Registros (filas): {df.shape[0]}")
    print(f"   - Variables (columnas): {df.shape[1]}\n")

    # b. Tipo de dato de cada variable
    print("b. Tipo de dato de cada variable:")
    for col in df.columns:
        print(f"   - {col}: {df[col].dtype}")
    print()

    # c. Valores faltantes
    print("c. Cantidad y porcentaje de valores faltantes por variable:")
    faltantes = df.isna().sum()
    total = len(df)
    for col in df.columns:
        nulos = faltantes[col]
        pct = (nulos / total) * 100
        print(f"   - {col}: {nulos} ({pct:.1f}%)")
    print()

    # d. Valores únicos por variable
    print("d. Cantidad de valores únicos por variable:")
    for col in df.columns:
        print(f"   - {col}: {df[col].nunique()}")
    print()

    # e. Registros duplicados exactos
    duplicados_exactos = df.duplicated().sum()
    print(f"e. Registros duplicados exactos: {duplicados_exactos}\n")

    # f. Variables con valores fuera de dominio o inconsistentes
    print("f. Variables con valores fuera de dominio o inconsistentes:")
    # Para columnas categóricas clave, comparar con listas esperadas
    dominios_esperados = {
        "DEPARTAMENTO": sorted(set(DEPARTAMENTOS) | {"ALTA VERAPAZ", "BAJA VERAPAZ", "CIUDAD CAPITAL"}),
        "NIVEL": ["BASICO", "DIVERSIFICADO", "PRIMARIA", "PARVULOS", "PREPRIMARIA BILINGUE", "PRIMARIA DE ADULTOS"],
        "SECTOR": ["OFICIAL", "PRIVADO", "MUNICIPAL", "COOPERATIVA"],
        "AREA": ["URBANA", "RURAL"],
        "STATUS": ["ABIERTA", "CERRADA DEFINITIVAMENTE", "CERRADA TEMPORALMENTE", "TEMPORAL NOMBRAMIENTO"],
        "MODALIDAD": ["MONOLINGUE", "BILINGUE"],
        "JORNADA": ["MATUTINA", "VESPERTINA", "NOCTURNA", "DOBLE", "SIN JORNADA"],
        "PLAN": ["DIARIO(REGULAR)", "SABATINO", "DOMINICAL", "FIN DE SEMANA", "IRREGULAR", "MIXTO", "A DISTANCIA", "VIRTUAL A DISTANCIA", "SEMIPRESENCIAL (FIN DE SEMANA)", "SEMIPRESENCIAL (UN DÍA A LA SEMANA)"]
    }
    for col, esperados in dominios_esperados.items():
        if col in df.columns:
            valores_reales = set(df[col].dropna().unique())
            fuera = valores_reales - set(esperados)
            if fuera:
                print(f"   - {col}: valores fuera del dominio esperado: {fuera}")
            else:
                print(f"   - {col}: todos los valores están dentro del dominio esperado.")

    # Otras columnas numéricas (p.ej., código, teléfono) se revisan en g.
    print()

    # g. Variables con formatos inconsistentes
    print("g. Variables con formatos inconsistentes:")
    # Código: formato esperado ##-##-####-## (puede variar)
    if "CODIGO" in df.columns:
        codigos = df["CODIGO"].dropna().astype(str)
        # Buscar códigos que no sigan el patrón típico
        import re
        patron = re.compile(r'^\d{2}-\d{2}-\d{4}-\d{2}$')
        inconsistentes = codigos[~codigos.str.match(patron)]
        if len(inconsistentes) > 0:
            print(f"   - CODIGO: {len(inconsistentes)} registros no siguen el formato ##-##-####-##")
            # mostrar algunos ejemplos
            print(f"     Ejemplos: {inconsistentes.head(3).tolist()}")
        else:
            print("   - CODIGO: todos los códigos siguen el formato esperado.")

    # Teléfono: debería contener solo dígitos, guiones o espacios
    if "TELEFONO" in df.columns:
        telefonos = df["TELEFONO"].dropna().astype(str)
        # eliminar espacios, guiones, paréntesis, etc.
        limpios = telefonos.str.replace(r'[\s\-\(\)]', '', regex=True)
        no_numericos = limpios[~limpios.str.isdigit()]
        if len(no_numericos) > 0:
            print(f"   - TELEFONO: {len(no_numericos)} registros contienen caracteres no numéricos.")
            print(f"     Ejemplos: {no_numericos.head(3).tolist()}")
        else:
            print("   - TELEFONO: todos los valores son numéricos (tras limpieza).")

    # Otras columnas podrían tener formatos mixtos (ej. fechas) pero no aplica aquí.
    print()

    # h. Identificación de problemas potenciales de calidad de datos
    print("h. Identificación de problemas potenciales de calidad de datos:")
    problemas = []
    # Alto porcentaje de faltantes en alguna columna (>50%)
    altos_faltantes = [col for col in df.columns if (df[col].isna().sum()/len(df))*100 > 50]
    if altos_faltantes:
        problemas.append(f"   - Columnas con más del 50% de datos faltantes: {altos_faltantes}")
    # Duplicados exactos
    if duplicados_exactos > 0:
        problemas.append(f"   - Existen {duplicados_exactos} filas duplicadas exactas.")
    # Valores inconsistentes detectados en dominios
    # Ya reportados arriba, los resumimos:
    if any(fuera for col, fuera in dominios_esperados.items() if col in df.columns):
        problemas.append("   - Algunas variables categóricas tienen valores fuera del dominio esperado (ver sección f).")
    # Formatos inconsistentes
    if "CODIGO" in df.columns and len(inconsistentes) > 0:
        problemas.append(f"   - {len(inconsistentes)} códigos tienen formato inesperado.")
    if "TELEFONO" in df.columns and len(no_numericos) > 0:
        problemas.append(f"   - {len(no_numericos)} teléfonos contienen caracteres no numéricos.")

    if problemas:
        for p in problemas:
            print(p)
    else:
        print("   - No se detectaron problemas graves de calidad de datos.")
    print("\n" + "="*80 + "\n")

# ============================================================================
# FLUJO PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    # 1. Verificar si existe el CSV unificado
    if os.path.exists(CSV_SALIDA):
        respuesta = input(f"Ya existe el archivo '{CSV_SALIDA}'. ¿Desea usarlo para el análisis sin descargar nuevamente? (s/n): ").strip().lower()
        if respuesta == 's' or respuesta == 'si':
            print("Cargando archivo existente...")
            df = pd.read_csv(CSV_SALIDA, encoding='utf-8-sig')
            print(f"Archivo cargado: {len(df)} filas, {len(df.columns)} columnas.")
        else:
            print("Procediendo con descarga nueva...")
            df = descargar_y_unificar()
    else:
        print("No se encontró archivo CSV unificado. Procediendo con descarga...")
        df = descargar_y_unificar()

    # 2. Ejecutar diagnóstico sobre el DataFrame (sea cargado o recién descargado)
    if df is not None and not df.empty:
        diagnosticar_datos(df)
    else:
        print("No hay datos para analizar.")

Cargando archivo existente...
Archivo cargado: 27413 filas, 19 columnas.

DIAGNÓSTICO DE CALIDAD DE DATOS

a. Número de registros y variables:
   - Registros (filas): 27413
   - Variables (columnas): 19

b. Tipo de dato de cada variable:
   - CODIGO: object
   - DISTRITO: object
   - DEPARTAMENTO: object
   - MUNICIPIO: object
   - ESTABLECIMIENTO: object
   - DIRECCION: object
   - TELEFONO: object
   - SUPERVISOR: object
   - DIRECTOR: object
   - NIVEL: object
   - SECTOR: object
   - AREA: object
   - STATUS: object
   - MODALIDAD: object
   - JORNADA: object
   - PLAN: object
   - DEPARTAMENTAL: object
   - DEPARTAMENTO_CONSULTADO: object
   - NIVEL_CONSULTADO: object

c. Cantidad y porcentaje de valores faltantes por variable:
   - CODIGO: 0 (0.0%)
   - DISTRITO: 1616 (5.9%)
   - DEPARTAMENTO: 0 (0.0%)
   - MUNICIPIO: 0 (0.0%)
   - ESTABLECIMIENTO: 7 (0.0%)
   - DIRECCION: 279 (1.0%)
   - TELEFONO: 2800 (10.2%)
   - SUPERVISOR: 1620 (5.9%)
   - DIRECTOR: 4634 (16.9%)
   - NIVEL: 